# Qwen-Image-Edit-2509 Inference Benchmark

This notebook benchmarks the inference performance of the Qwen-Image-Edit-2509 model on a single Nvidia V100 GPU.

In [ ]:
import os
import time
import torch
import polars as pl
from PIL import Image
from diffusers import QwenImageEditPlusPipeline

## Load Model and Input Images

First, we'll load the Qwen-Image-Edit-2509 pipeline and move it to the GPU. We'll also load the input images needed for the benchmark.

In [ ]:
# Load pipeline
pipeline = QwenImageEditPlusPipeline.from_pretrained("Qwen/Qwen-Image-Edit-2509", torch_dtype=torch.bfloat16)
pipeline.to('cuda')
pipeline.set_progress_bar_config(disable=True)
print("Pipeline loaded on CUDA")

# You need to provide input1.png and input2.png in the working directory
# Or modify the paths below to point to your images
image1 = Image.open("input1.png")
image2 = Image.open("input2.png")
print("Input images loaded")

## Configure Inference Parameters

Next, we'll set up the parameters for the image editing inference.

In [ ]:
# Configuration
num_runs = 5  # Number of inference runs for benchmarking
prompt = "The magician bear is on the left, the alchemist bear is on the right, facing each other in the central park square."

# Prepare inputs
inputs = {
    "image": [image1, image2],
    "prompt": prompt,
    "generator": torch.manual_seed(0),
    "true_cfg_scale": 4.0,
    "negative_prompt": " ",
    "num_inference_steps": 40,
    "guidance_scale": 1.0,
    "num_images_per_prompt": 1,
}

## Run Inference Benchmark

Now we'll run the inference multiple times and measure the performance.

In [ ]:
# Benchmark inference
timing_results = []
print(f"Running inference {num_runs} times...")

for i in range(num_runs):
    start_time = time.time()
    with torch.inference_mode():
        output = pipeline(**inputs)
    end_time = time.time()
    
    elapsed_time = end_time - start_time
    timing_results.append(elapsed_time)
    print(f"Run {i+1}: {elapsed_time:.2f} seconds")

## Collect and Analyze Performance Metrics

Finally, we'll collect the timing results into a polars DataFrame and calculate performance statistics.

In [ ]:
# Collect metrics
df = pl.DataFrame({
    "run": range(1, num_runs + 1),
    "inference_time_seconds": timing_results
})

# Display detailed results
print("Detailed timing results:")
print(df)

# Calculate statistics
mean_time = df["inference_time_seconds"].mean()
std_time = df["inference_time_seconds"].std()
min_time = df["inference_time_seconds"].min()
max_time = df["inference_time_seconds"].max()

print("\nBenchmark Results:")
print(f"Mean inference time: {mean_time:.2f} seconds")
print(f"Standard deviation: {std_time:.2f} seconds")
print(f"Min inference time: {min_time:.2f} seconds")
print(f"Max inference time: {max_time:.2f} seconds")

## Save Results

We'll save both the edited image and the benchmark results to files.

In [ ]:
# Save output image from last run
output_image = output.images[0]
output_image.save("output_image_edit_plus.png")
print(f"Output image saved at {os.path.abspath('output_image_edit_plus.png')}")

# Save metrics
metrics_path = "benchmark_results.csv"
df.write_csv(metrics_path)
print(f"Detailed results saved to {os.path.abspath(metrics_path)}")